[![Homepage](https://img.shields.io/badge/homepage-blueviolet?logo=htmx)](https://www.bendai.org/CUHK-STAT3009/)
[![GitHub](https://img.shields.io/badge/GitHub-black.svg?logo=github)](https://github.com/statmlben/CUH-STAT3009)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/statmlben/CUHK-STAT3009/blob/main/nb/STAT3009_NN.ipynb)

# Notebook 06: Neural networks

**STAT3009 · Recommender Systems**  
**Suggested time:** 120–150 minutes of guided coding and discussion

The linear model used one fixed transformation, $x^\top w+b$. A neural network replaces it with a **composition of learned linear maps and fixed nonlinear activations**. The objective does not change: we still minimize an average loss over training rows.

This notebook uses **skorch**, a thin sklearn-style wrapper around PyTorch. It gives us the Keras-like calls `model.fit(X, y)` and `model.predict(X)`. The slides show the `zero_grad → backward → step` loop that skorch runs underneath.

This notebook follows the lecture's four implementation modules:

```text
Data  →  Model  →  Loss  →  Optimization  →  Evaluation
```

By the end, you should be able to:

1. write the training objective and distinguish the loss from a reporting metric;
2. build a network with `nn.Linear`, `nn.ReLU`, `nn.Sequential`, and `nn.Module`;
3. check the shape contract between consecutive layers;
4. configure and call skorch's `.fit`, and explain the batch, epoch, learning rate, and SGD update inside it;
5. train with skorch, inspect the validation curve, and report test RMSE and MAE.

## 1. Setup

We use the course's fixed California-housing split, PyTorch for the network, skorch for the high-level `.fit` API, and scikit-learn for the baseline and reporting metrics. The dataset is small enough that CPU training is fast.

In Colab, PyTorch is preinstalled; install skorch once with `!pip install skorch`. On a local machine, install both with `pip install torch scikit-learn skorch`.

In [ ]:
# In Colab, uncomment the next line if skorch is not installed:
# !pip install skorch

import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import skorch
import torch
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
)
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from skorch import NeuralNetRegressor
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

pd.set_option('display.max_rows', 12)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
plt.rcParams['figure.dpi'] = 110

torch.manual_seed(0)
np.random.seed(0)

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('skorch:', skorch.__version__)
print('scikit-learn:', sklearn.__version__)
print('NumPy:', np.__version__)

## 2. Recall: the linear-regression objective

The target `MedHouseVal` is measured in units of $100{,}000$. We keep the published test target untouched until the final evaluation. skorch holds out part of the training rows for validation, so we do not create a separate validation split by hand.

$$
\widehat y_i=x_i^\top w+b,
\qquad
J(w,b)=\frac{1}{n}\sum_{i=1}^{n}\bigl(y_i-x_i^\top w-b\bigr)^2.
$$

The model is one linear transformation. Training chooses $w,b$ to minimize the average squared error on the training set. Standardization is fitted on the training data, then applied to the test rows.

In [ ]:
HOUSING_BASE_URL = (
    'https://raw.githubusercontent.com/'
    'statmlben/CUHK-STAT3009/main/dataset/housing'
)

housing_train = pd.read_csv(f'{HOUSING_BASE_URL}/train.csv')

feature_columns = [
    'MedInc', 'HouseAge', 'AveRooms', 'AveBedrms',
    'Population', 'AveOccup', 'Latitude', 'Longitude',
]
target_column = 'MedHouseVal'

X_train = housing_train[feature_columns].to_numpy(dtype=np.float32)
y_train = housing_train[[target_column]].to_numpy(dtype=np.float32)

# Load only the test features now. The test target is opened at final evaluation.
housing_test_features = pd.read_csv(
    f'{HOUSING_BASE_URL}/test.csv', usecols=feature_columns
)
X_test = housing_test_features.to_numpy(dtype=np.float32)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

BATCH_SIZE = 64
X_train_t = torch.from_numpy(X_train)
y_train_t = torch.from_numpy(y_train)
# The tensors above feed the pure-PyTorch example below.

print('training rows:', X_train.shape, y_train.shape)
print('test rows (labels not loaded yet):', X_test.shape)
display(housing_train.head())

| component | choice in this example |
|---|---|
| Data | 8 numerical features; target is median house value |
| Model | linear regression, then a neural network |
| Loss | mean squared error, used by the optimizer |
| Metrics | RMSE and MAE, reported for interpretation |

The loss is what training minimizes. A metric is what we report; they need not be the same function.

In [ ]:
baseline = LinearRegression().fit(X_train, y_train)
baseline_test_pred = baseline.predict(X_test)

print('fitted coefficients:', baseline.coef_.round(3))
print('test predictions computed; test RMSE is reported at the end.')

### Checkpoint

1. Which values are **learned** from the training rows?
2. Which values are **chosen** before fitting?
3. Why must the scaler be fitted on the training data rather than on the test rows?
4. What does one entry of `baseline.coef_` mean when the corresponding feature is standardized?

## 3. From linear to nonlinear

Without a nonlinearity, two linear layers collapse into one:

$$
W_2(W_1x+b_1)+b_2=(W_2W_1)x+(W_2b_1+b_2).
$$

A nonlinear activation between the layers prevents this collapse. We use ReLU only:

$$
A(z)=\max(0,z).
$$

A two-layer network is therefore

$$
h=A(W_1x+b_1),
\qquad
\widehat y=W_2h+b_2.
$$

The objective is unchanged. We only make the prediction rule $f_\theta$ more flexible.

In [ ]:
torch.manual_seed(0)

W1 = torch.randn(32, 8)
b1 = torch.randn(32)
W2 = torch.randn(1, 32)
b2 = torch.randn(1)
x = torch.randn(8)

h = W1 @ x + b1
y_two_layers = W2 @ h + b2

W_equivalent = W2 @ W1
b_equivalent = W2 @ b1 + b2
y_one_layer = W_equivalent @ x + b_equivalent

print('two linear layers:', y_two_layers.item())
print('one equivalent linear layer:', y_one_layer.item())
print('same function:', torch.allclose(y_two_layers, y_one_layer))

## 4. The data module: examples become batches

skorch builds the mini-batches and the validation split for us. We only choose `batch_size=64` and `iterator_train__shuffle=True`.

| term | meaning |
|---|---|
| loss $L$ | error for one prediction |
| objective $J$ | average loss over the training rows |
| gradient | derivative of the objective with respect to the parameters |
| batch $B$ | the subset of rows used for one update |
| batch size $b$ | number of rows in that batch |
| step / iteration | one gradient update, using one batch |
| epoch | one pass over the training set; about $n/b$ updates |
| learning rate $\eta$ | step size in the parameter update |

The training set has $n=13{,}828$ rows and $b=64$. skorch holds out one fifth for validation, so one training epoch uses about 173 batches.

In [ ]:
train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=BATCH_SIZE,
    shuffle=True,
)

xb, yb = next(iter(train_loader))
print('one batch:', tuple(xb.shape), tuple(yb.shape))
print('batches per epoch:', len(train_loader))

## 5. The model module: build with bricks

Think of each layer as a **brick** with an input shape and an output shape. `nn.Linear` is the only brick that learns weights in our models; `nn.ReLU` reshapes each number without changing the tensor shape.

### Inside one `nn.Linear` brick

`nn.Linear(d_in, d_out)` stores a weight matrix and a bias vector:

$$
h=xW^\top+b.
$$

| object | shape |
|---|---|
| input $x$ | $(B,d_{\mathrm{in}})$ |
| weight $W$ | $(d_{\mathrm{out}},d_{\mathrm{in}})$ |
| bias $b$ | $(d_{\mathrm{out}},)$ |
| output $h$ | $(B,d_{\mathrm{out}})$ |

The bias is broadcast across the batch. A `Linear(8, 1)` layer has $8\cdot1+1=9$ learned numbers and is exactly linear regression.

In [ ]:
one_layer = nn.Linear(8, 1)

print(one_layer)
print('weight shape:', tuple(one_layer.weight.shape))
print('bias shape:  ', tuple(one_layer.bias.shape))
print('learned numbers:', sum(p.numel() for p in one_layer.parameters()))
print('output shape:', tuple(one_layer(X_train_t[:5]).shape))

manual_output = X_train_t[:5] @ one_layer.weight.T + one_layer.bias
print('same as the formula:', torch.allclose(one_layer(X_train_t[:5]), manual_output))

### Stack bricks with `nn.Sequential`

Our running example is

```text
Linear(8,32) → ReLU → Linear(32,16) → ReLU → Linear(16,1)
```

| brick | output shape | learned numbers |
|---|---|---|
| `Linear(8,32)` | `(B,32)` | $8\cdot32+32=288$ |
| `ReLU()` | `(B,32)` | 0 |
| `Linear(32,16)` | `(B,16)` | $32\cdot16+16=528$ |
| `ReLU()` | `(B,16)` | 0 |
| `Linear(16,1)` | `(B,1)` | $16\cdot1+1=17$ |

`Sequential` is the baseplate: it calls the bricks in order. The whole model has **833** learned numbers.

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        torch.manual_seed(0)  # pin initialization so CV and refit are comparable
        self.net = nn.Sequential(
            nn.Linear(8, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.net(x)


torch.manual_seed(0)
mlp = MLP()
print(mlp)
print('total learned numbers:', sum(p.numel() for p in mlp.parameters()))
print('output shape:', tuple(mlp(X_train_t[:5]).shape))

### Build a deeper stack

Stack the bricks explicitly and check the output width at every step. This is the practice architecture in miniature.

| brick | output shape | learned numbers |
|---|---|---|
| `Linear(8,64)` | `(B,64)` | $8\cdot64+64=576$ |
| `ReLU()` | `(B,64)` | 0 |
| `Linear(64,32)` | `(B,32)` | $64\cdot32+32=2080$ |
| `ReLU()` | `(B,32)` | 0 |
| `Linear(32,16)` | `(B,16)` | $32\cdot16+16=528$ |
| `ReLU()` | `(B,16)` | 0 |
| `Linear(16,1)` | `(B,1)` | $16\cdot1+1=17$ |

The whole model has **3201** learned numbers.

In [ ]:
deep_model = nn.Sequential(
    nn.Linear(8, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 16), nn.ReLU(),
    nn.Linear(16, 1),
)

print(deep_model)
print('total learned numbers:',
      sum(p.numel() for p in deep_model.parameters()))
print('output shape:', tuple(deep_model(X_train_t[:5]).shape))

### Branch and merge with `nn.Module`

`Sequential` can only express a straight line of bricks. When the forward pass **branches and merges**, define a custom `nn.Module`:

```python
class ParallelMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.up = ...
        self.down = ...

    def forward(self, x):
        return self.up(x) + self.down(x)
```

- `__init__` registers the bricks.
- `forward` defines the path from input to output.
- PyTorch calls `forward` automatically when you write `model(x)`.

The next model computes

$$
f(x)=\psi_{[3,1]}(x)+\psi_{[2,1]}(x),
$$

where each $\psi$ is a small ReLU network.

In [ ]:
class ParallelMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.up = nn.Sequential(
            nn.Linear(8, 3), nn.ReLU(),
            nn.Linear(3, 1),
        )
        self.down = nn.Sequential(
            nn.Linear(8, 2), nn.ReLU(),
            nn.Linear(2, 1),
        )

    def forward(self, x):
        up = self.up(x)
        down = self.down(x)
        return up + down


torch.manual_seed(0)
parallel_model = ParallelMLP()
print(parallel_model)
print('total learned numbers:', sum(p.numel() for p in parallel_model.parameters()))
print('output shape:', tuple(parallel_model(X_train_t[:5]).shape))

### Concatenate two branches

The `ParallelMLP` added two scalar outputs. Here each branch produces a **4-dimensional feature vector**, and `torch.cat` joins them into 8 features for the final head. This is the shape pattern NCF will use.

```text
x → Linear(8,4) → ReLU ─┐
                         ├─ concat (B,8) → Linear(8,1) → ŷ
x → Linear(8,4) → ReLU ─┘
```

In [ ]:
class ConcatMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.left = nn.Sequential(
            nn.Linear(8, 4), nn.ReLU(),
        )
        self.right = nn.Sequential(
            nn.Linear(8, 4), nn.ReLU(),
        )
        self.head = nn.Linear(8, 1)

    def forward(self, x):
        left = self.left(x)
        right = self.right(x)
        return self.head(torch.cat([left, right], dim=1))


concat_model = ConcatMLP()
print(concat_model)
print('total learned numbers:',
      sum(p.numel() for p in concat_model.parameters()))

left = concat_model.left(X_train_t[:5])
right = concat_model.right(X_train_t[:5])
joined = torch.cat([left, right], dim=1)

print('left shape:  ', tuple(left.shape))
print('right shape: ', tuple(right.shape))
print('joined shape:', tuple(joined.shape))
print('output shape:', tuple(concat_model(X_train_t[:5]).shape))

### Parameters versus hyperparameters

| | examples | who decides? |
|---|---|---|
| **Parameters** $\theta$ | all `weight` and `bias` entries | the optimizer, using training rows |
| **Hyperparameters** | depth, width, activation, learning rate, batch size, epochs | we choose; validation compares candidates |

Architecture choices define the model family. Training only changes the parameters inside it.

### The shape contract

Before training, check every brick's output shape against the next brick's input shape. The first dimension is the batch size; the second is the feature width.

A mismatch such as `Linear(8,32)` followed directly by `Linear(16,1)` fails at the first forward pass:

```text
32 ≠ 16  →  mat1 and mat2 shapes cannot be multiplied
```

In [ ]:
bad_model = nn.Sequential(
    nn.Linear(8, 32),
    nn.Linear(16, 1),
)

try:
    bad_model(X_train_t[:4])
except RuntimeError as error:
    print('RuntimeError:', str(error).split('.')[0])

## 6. The loss module: what do we minimize?

Training minimizes the mean squared error:

$$
J(\theta)=\frac{1}{n}\sum_{i=1}^{n}\bigl(y_i-f_\theta(x_i)\bigr)^2.
$$

For reporting, we use the same units as the target:

$$
\mathrm{RMSE}=\sqrt{\frac1n\sum_i(y_i-\widehat y_i)^2},
\qquad
\mathrm{MAE}=\frac1n\sum_i\lvert y_i-\widehat y_i\rvert.
$$

MSE is sensitive to large errors and is convenient for optimization. RMSE and MAE are easier to interpret. skorch uses `nn.MSELoss` as the criterion; we use scikit-learn's RMSE and MAE for reporting.

In [ ]:
baseline_train_pred = baseline.predict(X_train)
baseline_mse = mean_squared_error(y_train, baseline_train_pred)

print('baseline training MSE:', f'{baseline_mse:.4f}')
print('baseline training RMSE:', f'{root_mean_squared_error(y_train, baseline_train_pred):.4f}')
print('baseline training MAE: ', f'{mean_absolute_error(y_train, baseline_train_pred):.4f}')

## 7. The optimization module: from PyTorch to skorch

Gradient descent repeatedly moves opposite the gradient:

$$
\theta^{(t+1)}=\theta^{(t)}-\eta\nabla_\theta J\bigl(\theta^{(t)}\bigr).
$$

A full-batch step uses all $n$ rows. **Mini-batch SGD** uses one batch $B$:

$$
\nabla J_B(\theta)=\frac{1}{\lvert B\rvert}\sum_{j\in B}\nabla_\theta L\bigl(y_j,f_\theta(x_j)\bigr),
\qquad
\theta\leftarrow\theta-\eta\nabla J_B(\theta).
$$

The batch gradient is noisy but cheap; one epoch gives us about $n/b$ updates instead of one.

skorch is a thin sklearn-style wrapper around PyTorch. It builds the mini-batches and runs this update for us. We still choose the same pieces: model, loss, optimizer, learning rate, batch size, and number of epochs.

### Train with pure PyTorch

First write the loop by hand. Each iteration uses one mini-batch, so the gradient is a batch estimate of the full gradient. `loss.backward()` applies autograd, and `optimizer.step()` performs the SGD update.

```text
zero_grad → forward → loss → backward → step
```

The next subsection shows the same loop wrapped by skorch.

In [ ]:
torch.manual_seed(0)
torch_model = MLP()
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(torch_model.parameters(), lr=0.01)

for epoch in range(5):
    torch_model.train()
    epoch_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(torch_model(xb), yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f'epoch {epoch + 1}: training MSE {epoch_loss / len(train_loader):.4f}')

### What skorch's `.fit` does inside

The loop above is exactly what skorch runs for every epoch. `max_epochs`, `batch_size`, `lr`, `optimizer`, and `criterion` configure it; we do not write the loop again.

### Wrap the model for skorch

`NeuralNetRegressor` expects a **model class**, not a model instance. We defined `MLP`; its output shape `(B,1)` matches the target shape `(n,1)`.

### Fit the model and watch validation

We train for a fixed number of epochs and record the training and validation history. The validation curve tells us whether more epochs help or whether the model is starting to overfit.

In [ ]:
torch.manual_seed(0)
model = NeuralNetRegressor(
    module=MLP,
    criterion=nn.MSELoss,
    optimizer=torch.optim.SGD,
    max_epochs=100,
    lr=0.01,
    batch_size=BATCH_SIZE,
    iterator_train__shuffle=True,
    verbose=0,
)

model.fit(X_train, y_train)

print('epochs run:', len(model.history))
print('final training RMSE:',
      f"{np.sqrt(model.history[-1, 'train_loss']):.4f}")
print('final validation RMSE:',
      f"{np.sqrt(model.history[-1, 'valid_loss']):.4f}")

In [ ]:
epochs = np.arange(1, len(model.history) + 1)

plt.figure(figsize=(8, 4))
plt.plot(epochs, np.sqrt(model.history[:, 'train_loss']), label='training RMSE')
plt.plot(epochs, np.sqrt(model.history[:, 'valid_loss']), label='validation RMSE')
plt.xlabel('epoch')
plt.ylabel('RMSE (target units)')
plt.title('Training and validation curves')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

The training curve keeps falling, while the validation curve can flatten or rise. We train for a fixed number of epochs; the validation curve tells us whether that number is sensible. The test target is still untouched.

## 8. Tune hyperparameters with GridSearchCV

**`GridSearchCV`** compares hyperparameter settings with cross-validation, using the same sklearn pattern as the earlier notebooks. skorch models are sklearn estimators, so this works directly.

We turn off skorch's internal validation split (`train_split=False`) because each CV fold supplies its own validation data. The regression score is **negative RMSE** because sklearn always maximizes a score.

In [ ]:
search_net = NeuralNetRegressor(
    module=MLP,
    criterion=nn.MSELoss,
    optimizer=torch.optim.SGD,
    batch_size=BATCH_SIZE,
    iterator_train__shuffle=True,
)

# GridSearchCV supplies the validation folds, so skorch does not split again.
search_net.set_params(train_split=False, verbose=0)

param_grid = {
    'lr': [0.01],
    'max_epochs': [80, 100],
    'batch_size': [32, 64],
}

torch.manual_seed(0)
gs = GridSearchCV(
    search_net,
    param_grid,
    refit=True,
    cv=3,
    scoring='neg_root_mean_squared_error',
    verbose=1,
)
gs.fit(X_train, y_train)

print('best CV RMSE:', f'{-gs.best_score_:.4f}')
print('best params:', gs.best_params_)

cv_results = pd.DataFrame(gs.cv_results_)[
    ['param_lr', 'param_max_epochs', 'param_batch_size',
     'mean_test_score', 'rank_test_score']
].copy()
cv_results['validation_RMSE'] = -cv_results.pop('mean_test_score')
display(cv_results.sort_values('validation_RMSE'))

`refit=True` refits the best settings on all training rows and stores the fitted model in `gs.best_estimator_`. We use that tuned model for the final test report.

### Final evaluation on the test set

Only now do we open the test target. `final_net = gs.best_estimator_` is the winning model refit on all training rows. The validation row reports the three-fold CV RMSE; the test row is the final check.

In [ ]:
housing_test_labels = pd.read_csv(
    f'{HOUSING_BASE_URL}/test.csv', usecols=[target_column]
)
y_test = housing_test_labels[[target_column]].to_numpy(dtype=np.float32)

final_net = gs.best_estimator_
train_pred = final_net.predict(X_train)
test_pred = final_net.predict(X_test)

results = pd.DataFrame({
    'split': ['training', 'validation (3-fold CV)', 'test'],
    'RMSE': [
        root_mean_squared_error(y_train, train_pred),
        -gs.best_score_,
        root_mean_squared_error(y_test, test_pred),
    ],
})

display(results)
print('training MAE:', f'{mean_absolute_error(y_train, train_pred):.4f}')
print('test MAE:    ', f'{mean_absolute_error(y_test, test_pred):.4f}')
print('linear-regression test RMSE:',
      f'{root_mean_squared_error(y_test, baseline.predict(X_test)):.4f}')

**Discuss:** How does the three-fold CV score from `GridSearchCV` compare with the fixed-epoch validation curve? Why might the two estimates differ, and why do we still open the test target only once?

## 9. In-class practice: build and diagnose a housing network

Use the same training and test data. Work in pairs for 20–25 minutes.

1. Build `PracticeMLP` with hidden widths 256 → 128 → 64 → 32 → 16.
2. Wrap it in `NeuralNetRegressor` with `nn.MSELoss`, `torch.optim.SGD`, learning rate `0.01`, `batch_size=64`, and `max_epochs=50`.
3. Fit, plot `practice_net.history`, and report test RMSE and MAE.

In [ ]:
class PracticeMLP(nn.Module):
    def __init__(self):
        super().__init__()
        torch.manual_seed(0)  # reproducible initialization
        self.net = nn.Sequential(
            nn.Linear(8, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.net(x)


practice_model = PracticeMLP()
print(practice_model)
print('total learned numbers:',
      sum(p.numel() for p in practice_model.parameters()))

In [ ]:
# Complete the estimator and fit:
# practice_net = NeuralNetRegressor(
#     module=PracticeMLP,
#     criterion=nn.MSELoss,
#     optimizer=torch.optim.SGD,
#     max_epochs=50,
#     lr=0.01,
#     batch_size=BATCH_SIZE,
#     iterator_train__shuffle=True,
#     verbose=0,
# )
# practice_net.fit(X_train, y_train)
# practice_pred = practice_net.predict(X_test)
# print('test RMSE:', f'{root_mean_squared_error(y_test, practice_pred):.4f}')
# print('test MAE: ', f'{mean_absolute_error(y_test, practice_pred):.4f}')

### Solution

In [ ]:
torch.manual_seed(0)
practice_net = NeuralNetRegressor(
    module=PracticeMLP,
    criterion=nn.MSELoss,
    optimizer=torch.optim.SGD,
    max_epochs=50,
    lr=0.01,
    batch_size=BATCH_SIZE,
    iterator_train__shuffle=True,
    verbose=0,
)

practice_net.fit(X_train, y_train)
practice_pred = practice_net.predict(X_test)
print('total learned numbers:',
      sum(p.numel() for p in practice_net.module_.parameters()))
print('final validation RMSE:',
      f"{np.sqrt(practice_net.history[-1, 'valid_loss']):.4f}")
print('test RMSE:', f'{root_mean_squared_error(y_test, practice_pred):.4f}')
print('test MAE: ', f'{mean_absolute_error(y_test, practice_pred):.4f}')

practice_epochs = np.arange(1, len(practice_net.history) + 1)
plt.figure(figsize=(8, 4))
plt.plot(practice_epochs, np.sqrt(practice_net.history[:, 'train_loss']),
         label='training RMSE')
plt.plot(practice_epochs, np.sqrt(practice_net.history[:, 'valid_loss']),
         label='validation RMSE')
plt.xlabel('epoch')
plt.ylabel('RMSE (target units)')
plt.title('Practice model: wider and deeper')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

**Discuss:** Compare the practice model with the smaller `MLP` above. Did the larger network improve validation RMSE? What does its validation curve suggest?

## 10. Failure patterns and rules of thumb

| symptom | likely cause | next check |
|---|---|---|
| loss oscillates | learning rate too large | reduce the step size |
| loss hardly moves | learning rate too small or inputs poorly scaled | inspect features and gradients |
| training improves, validation worsens | overfitting | simpler model or more data |
| validation looks excellent | possible leakage | audit the split and preprocessing |
| loss becomes `NaN` | unstable update or unscaled inputs | reduce the step size; standardize features |
| loss looks wrong but no error | prediction and target shapes differ | print `pred.shape` and `y.shape` |
| shape error at the first batch | consecutive bricks disagree | print the output shape of each brick |

| choice | starting point |
|---|---|
| output layer | linear for regression; sigmoid or softmax for classification |
| hidden layers | first layer about half the input width; halve thereafter |
| activation | ReLU is a good default |
| epochs | start with 20 to check convergence; then use 100 and choose by validation |
| batch size | powers of two; start with 16–64 |
| inputs | standardize numerical features before fitting |

These are starting points, not laws. Tune systematically and let validation choose.

## Exit ticket

Answer each question in one or two sentences.

1. What does `model.fit(X, y)` do internally? Name the five operations for one batch.
2. What do `max_epochs`, `batch_size`, and `lr` control?
3. Why do we keep a validation curve if we train for a fixed number of epochs?
4. Why do we keep the target shape `(n,1)` to match the model output?
5. Why do we let skorch hold out validation data, and why do we still keep the test target untouched until the end?
6. What is the difference between the `criterion` and the reporting metrics RMSE and MAE?
7. What does `model.predict(X)` return, and how do we score it?

8. In `GridSearchCV`, what do `cv=3`, `param_grid`, and `scoring='neg_root_mean_squared_error'` specify?
9. What does `refit=True` store in `gs.best_estimator_`?

## Summary

```text
X, y
  → model class: nn.Module, forward returns (B,)
  → NeuralNetRegressor(module, criterion, optimizer, max_epochs, batch_size, lr)
  → model.fit(X_train, y_train)       # inside: zero_grad → forward → loss → backward → step
  → model.history                 # training and validation curves
  → GridSearchCV(search_net, param_grid, cv=3, scoring='neg_root_mean_squared_error')
  → gs.best_estimator_          # tuned model refit on all training rows
  → model.predict(X_test)         # RMSE / MAE
```

A network is still a prediction rule with parameters, a loss, and an optimizer. skorch gives us a Keras-like `.fit`, while the underlying PyTorch update is still the five-line SGD step from the slides. Check shapes before training, check validation while training, and check the test set once at the end.

Next, we replace the numeric housing features with user and movie IDs, learn embedding tables for them, and combine the branches in a custom `nn.Module`. That is the neural collaborative filtering model.

## References

- PyTorch `nn.Linear`: <https://pytorch.org/docs/stable/generated/torch.nn.Linear.html>
- PyTorch `nn.Module`: <https://pytorch.org/docs/stable/generated/torch.nn.Module.html>
- PyTorch data loading: <https://pytorch.org/tutorials/beginner/basics/data_tutorial.html>
- PyTorch SGD: <https://pytorch.org/docs/stable/generated/torch.optim.SGD.html>
- scikit-learn California housing: <https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_california_housing.html>
- skorch `NeuralNetRegressor`: <https://skorch.readthedocs.io/en/stable/regressor.html>
